<a href="https://colab.research.google.com/github/Gaenariya/Kimnaryeong/blob/main/03_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 03. 전이학습(Transfer Learning) 이미지 분류 - 고양이 vs 개
# Google Colab에 그대로 복사해서 셀 단위(# %%)로 실행하면 됩니다.

# %%
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# %% 데이터셋 다운로드 (cats_vs_dogs, TF Datasets 사용)
import tensorflow_datasets as tfds

(raw_train, raw_val), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:90%]"],
    with_info=True,
    as_supervised=True,
)

IMG_SIZE = 160

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

train_ds = raw_train.map(preprocess).shuffle(1000).batch(32).prefetch(1)
val_ds = raw_val.map(preprocess).batch(32).prefetch(1)

# %% 샘플 이미지 확인
for img, label in train_ds.take(1):
    plt.imshow(img[0])
    plt.title("Cat" if label[0] == 0 else "Dog")
    plt.show()

# %% 사전학습된 MobileNetV2를 베이스로 사용 (전이학습)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False  # 베이스 모델은 학습하지 않고 고정

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])
model.summary()

# %% 학습 (베이스 모델 고정 상태로 빠르게 학습)
history = model.fit(train_ds, validation_data=val_ds, epochs=3)

# %% 학습 곡선
plt.plot(history.history["accuracy"], label="train_acc")
plt.plot(history.history["val_accuracy"], label="val_acc")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.show()

# %% 예측 예시
for img, label in val_ds.take(1):
    pred = model.predict(img)
    plt.imshow(img[0])
    plt.title(f"예측: {'Dog' if pred[0] > 0.5 else 'Cat'} (정답: {'Dog' if label[0]==1 else 'Cat'})")
    plt.show()

AttributeError: module 'tensorflow_datasets' has no attribute 'load'